# Monthly Analysis: Train.csv vs Test.csv

This notebook analyzes the monthly patterns in features to compare training and test datasets. 
We will:
1. Load both datasets
2. Handle missing values (-9999 in test data)
3. Extract features with monthly patterns
4. Compute monthly statistics for training data (by class: 0=no aquaculture, 1=aquaculture, and combined) and test data
5. Visualize monthly trends using error bar plots and box plots

In [ ]:

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


In [ ]:

# Load the datasets
print("Loading datasets...")
train_df = pd.read_csv('../data/Train.csv')
test_df = pd.read_csv('../data/Test.csv')

print(f"Training dataset shape: {train_df.shape}")
print(f"Test dataset shape: {test_df.shape}")

# Display column names to understand structure
print(f"Training columns: {list(train_df.columns)}")
print(f"Test columns: {list(test_df.columns)}")


In [ ]:
# Check for missing values (-9999 in test data)
print("Checking for missing value indicators...")

# Count -9999 values in test data
test_neg_9999_count = (test_df == -9999).sum().sum()
print(f"Total -9999 values in test data: {test_neg_9999_count}")

# Check if -9999 exists in training data (should not)
train_neg_9999_count = (train_df == -9999).sum().sum()
print(f"Total -9999 values in train data: {train_neg_9999_count}")

# Show columns with -9999 in test data
test_neg_9999_cols = (test_df == -9999).sum()
test_9999_cols = test_neg_9999_cols[test_neg_9999_cols > 0]
print(f"\nColumns with -9999 values in test data ({len(test_neg_9999_cols)} columns):")
print(test_neg_9999_cols.head(10))  # Show first 10
if len(test_neg_9999_cols) > 10:
    print(f"... and {len(test_neg_9999_cols) - 10} more")

# Prepare data for monthly analysis
# For training data: we want to analyze both classes separately AND combined
# For test data: no label column

# Identify feature columns (excluding ID and label for train, just ID for test)
train_feature_cols = [col for col in train_df.columns if col not in ['ID', 'label']]
test_feature_cols = [col for col in test_df.columns if col != 'ID']

print(f"Number of feature columns in train: {len(train_feature_cols)}")
print(f"Number of feature columns in test: {len(test_feature_cols)}")

# Verify they match
if set(train_feature_cols) == set(test_feature_cols):
    print("✓ Feature columns match between train and test")
else:
    print("✗ Feature columns differ!")
    print(f"Train-only: {set(train_feature_cols) - set(test_feature_cols)}")
    print(f"Test-only: {set(test_feature_cols) - set(train_feature_cols)}")

# Create datasets for analysis
# Combined training data (both classes)
train_features_combined = train_df[train_feature_cols].copy()  # Includes both classes

# Training data for class 0 (no aquaculture)
train_features_class0 = train_df[train_df['label'] == 0][train_feature_cols].copy()

# Training data for class 1 (aquaculture)
train_features_class1 = train_df[train_df['label'] == 1][train_feature_cols].copy()

# Test data (no label column)
test_features = test_df[test_feature_cols].copy()

# Replace -9999 with NaN in test features
test_features = test_features.replace(-9999, np.nan)

print(f"Training features shape (combined): {train_features_combined.shape}")
print(f"Training features shape (class 0): {train_features_class0.shape}")
print(f"Training features shape (class 1): {train_features_class1.shape}")
print(f"Test features shape: {test_features.shape}")

# Check for missing values after conversion
print(f"Missing values in train features (combined): {train_features_combined.isnull().sum().sum()}")
print(f"Missing values in train features (class 0): {train_features_class0.isnull().sum().sum()}")
print(f"Missing values in train features (class 1): {train_features_class1.isnull().sum().sum()}")
print(f"Missing values in test features: {test_features.isnull().sum().sum()}")

In [ ]:
# Identify features with monthly patterns
# Features are expected to be in format: <base>_<MM> where MM is 01-12

# Pattern to capture base name and month
pattern = re.compile(r'^(.*)_(\d{2})$')

# Group features by base name
feature_groups = {}
for col in train_feature_cols:
    match = pattern.match(col)
    if match:
        base_name = match.group(1)
        month = match.group(2)
        if base_name not in feature_groups:
            feature_groups[base_name] = {}
        feature_groups[base_name][month] = col

# Filter to only include bases that have all 12 months
months_01_to_12 = {f"{i:02d}" for i in range(1, 13)}
complete_features = {}
for base, months_dict in feature_groups.items():
    if set(months_dict.keys()) == months_01_to_12:
        # Create ordered list of columns from month 01 to 12
        ordered_cols = [months_dict[f"{i:02d}"] for i in range(1, 13)]
        complete_features[base] = ordered_cols

print(f"Found {len(complete_features)} features with complete monthly data (01-12):")
for base in list(complete_features.keys())[:10]:  # Show first 10
    print(f"  {base}")
if len(complete_features) > 10:
    print(f"  ... and {len(complete_features) - 10} more")

# Also get features that might have partial monthly data (for completeness)
partial_features = {}
for base, months_dict in feature_groups.items():
    if set(months_dict.keys()) != months_01_to_12 and len(months_dict) > 0:
        partial_features[base] = months_dict

print(f"\nFound {len(partial_features)} features with partial monthly data.")
if len(partial_features) > 0:
    print(f"Examples: {list(partial_features.keys())[:5]}")

In [ ]:
# Compute monthly statistics for each feature
# We'll compute mean and standard error for each month (across samples)
# For training data: class 0, class 1, and combined
# For test data: as before

# Storage for results
# Structure: {feature_base: {'train_class0': {'mean': [], 'sem': []}, 'train_class1': {'mean': [], 'sem': []}, 'train_combined': {'mean': [], 'sem': []}, 'test': {'mean': [], 'sem': []}}}
monthly_stats = {}

months = [f"{i:02d}" for i in range(1, 13)]
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

print("Computing monthly statistics...")
for feature_base, column_list in complete_features.items():
    # Initialize storage for this feature
    monthly_stats[feature_base] = {
        'train_class0': {'mean': [], 'sem': []},
        'train_class1': {'mean': [], 'sem': []},
        'train_combined': {'mean': [], 'sem': []},
        'test': {'mean': [], 'sem': []}
    }
    
    # Process each month
    for month_idx, month in enumerate(months):
        col_name = column_list[month_idx]
        
        # Get training data for this month - class 0
        train_vals_class0 = train_features_class0[col_name].dropna().values
        # Get training data for this month - class 1
        train_vals_class1 = train_features_class1[col_name].dropna().values
        # Get training data for this month - combined
        train_vals_combined = train_features_combined[col_name].dropna().values
        # Get test data for this month
        test_vals = test_features[col_name].dropna().values
        
        # Compute statistics for train class 0
        if len(train_vals_class0) > 0:
            train_mean_0 = np.mean(train_vals_class0)
            train_sem_0 = np.std(train_vals_class0, ddof=1) / np.sqrt(len(train_vals_class0))
        else:
            train_mean_0 = np.nan
            train_sem_0 = np.nan
        
        # Compute statistics for train class 1
        if len(train_vals_class1) > 0:
            train_mean_1 = np.mean(train_vals_class1)
            train_sem_1 = np.std(train_vals_class1, ddof=1) / np.sqrt(len(train_vals_class1))
        else:
            train_mean_1 = np.nan
            train_sem_1 = np.nan
            
        # Compute statistics for train combined
        if len(train_vals_combined) > 0:
            train_mean_combined = np.mean(train_vals_combined)
            train_sem_combined = np.std(train_vals_combined, ddof=1) / np.sqrt(len(train_vals_combined))
        else:
            train_mean_combined = np.nan
            train_sem_combined = np.nan
        
        # Compute statistics for test
        if len(test_vals) > 0:
            test_mean = np.mean(test_vals)
            test_sem = np.std(test_vals, ddof=1) / np.sqrt(len(test_vals))
        else:
            test_mean = np.nan
            test_sem = np.nan
        
        # Store results
        monthly_stats[feature_base]['train_class0']['mean'].append(train_mean_0)
        monthly_stats[feature_base]['train_class0']['sem'].append(train_sem_0)
        monthly_stats[feature_base]['train_class1']['mean'].append(train_mean_1)
        monthly_stats[feature_base]['train_class1']['sem'].append(train_sem_1)
        monthly_stats[feature_base]['train_combined']['mean'].append(train_mean_combined)
        monthly_stats[feature_base]['train_combined']['sem'].append(train_sem_combined)
        monthly_stats[feature_base]['test']['mean'].append(test_mean)
        monthly_stats[feature_base]['test']['sem'].append(test_sem)
    
    # Progress indicator
    if list(complete_features.keys()).index(feature_base) % 5 == 0:
        print(f"  Processed {list(complete_features.keys()).index(feature_base) + 1}/{len(complete_features)} features")

print("Monthly statistics computation complete!")

In [ ]:
# Create error bar plots showing monthly trends for each feature
# Comparing train (class 0, class 1, combined) vs test

# Determine grid size for subplots
n_features = len(complete_features)
n_cols = 4
n_rows = (n_features + n_cols - 1) // n_cols  # Ceiling division

print(f"Creating error bar plots for {n_features} features...")
print(f"Using grid layout: {n_rows} rows × {n_cols} columns")

# Create figure
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

# Define colors
class0_color = '#2ca02c'  # Green
class1_color = '#d62728'  # Red
combined_color = '#1f77b4'  # Blue
test_color = '#ff7f0e'   # Orange

x_positions = np.arange(1, 13)  # Months 1-12

# Plot each feature
for idx, (feature_base, stats_dict) in enumerate(monthly_stats.items()):
    ax = axes[idx]
    
    # Get data
    train_mean_0 = np.array(stats_dict['train_class0']['mean'])
    train_sem_0 = np.array(stats_dict['train_class0']['sem'])
    train_mean_1 = np.array(stats_dict['train_class1']['mean'])
    train_sem_1 = np.array(stats_dict['train_class1']['sem'])
    train_mean_combined = np.array(stats_dict['train_combined']['mean'])
    train_sem_combined = np.array(stats_dict['train_combined']['sem'])
    test_mean = np.array(stats_dict['test']['mean'])
    test_sem = np.array(stats_dict['test']['sem'])
    
    # Plot train class 0 data
    ax.errorbar(x_positions, train_mean_0, yerr=train_sem_0,
                color=class0_color, label='Train (class 0 - no aquaculture)',
                marker='o', capsize=4, elinewidth=1.5, markeredgewidth=1.5)
    
    # Plot train class 1 data
    ax.errorbar(x_positions, train_mean_1, yerr=train_sem_1,
                color=class1_color, label='Train (class 1 - aquaculture)',
                marker='s', capsize=4, elinewidth=1.5, markeredgewidth=1.5)
    
    # Plot train combined data
    ax.errorbar(x_positions, train_mean_combined, yerr=train_sem_combined,
                color=combined_color, label='Train (combined)',
                marker='^', capsize=4, elinewidth=1.5, markeredgewidth=1.5)
    
    # Plot test data
    ax.errorbar(x_positions, test_mean, yerr=test_sem,
                color=test_color, label='Test',
                marker='D', capsize=4, elinewidth=1.5, markeredgewidth=1.5)
    
    # Customize subplot
    ax.set_title(f'{feature_base}', fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Month', fontsize=10)
    ax.set_ylabel('Value', fontsize=10)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(month_names, rotation=45, ha='right', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Add legend only to first subplot
    if idx == 0:
        ax.legend(loc='upper right', fontsize=9)
    
    # Set y-limits with padding
    all_vals = np.concatenate([train_mean_0, train_mean_1, train_mean_combined, test_mean])
    all_errs = np.concatenate([train_sem_0, train_sem_1, train_sem_combined, test_sem])
    valid_vals = all_vals[~np.isnan(all_vals)]
    valid_errs = all_errs[~np.isnan(all_errs)]
    if len(valid_vals) > 0:
        data_min = np.min(valid_vals - valid_errs)
        data_max = np.max(valid_vals + valid_errs)
        data_range = data_max - data_min
        if data_range > 0:
            ax.set_ylim(data_min - 0.05 * data_range, data_max + 0.05 * data_range)
    
    # Progress indicator
    if (idx + 1) % 5 == 0 or (idx + 1) == n_features:
        print(f"  Plotted {idx + 1}/{n_features} features")

# Hide unused subplots
for idx in range(len(monthly_stats), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Monthly Feature Trends: Train (Classes 0,1,Combined) vs Test (Error bars show standard error of the mean)',
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Create box plots to visualize distributions across months
# For a subset of features to avoid overly crowded plots

# Select a subset of features for box plotting (too many would be overwhelming)
max_features_for_boxplot = min(12, len(monthly_stats))  # Limit to 12 features
selected_features = list(monthly_stats.keys())[:max_features_for_boxplot]

print(f"Creating box plots for {len(selected_features)} features...")

# Determine grid size
n_cols = 4
n_rows = (len(selected_features) + n_cols - 1) // n_cols

fig2, axes2 = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
if n_rows == 1:
    axes2 = axes2.reshape(1, -1)
axes2 = axes2.flatten()

# Define colors for boxes (same as in error bar plots)
class0_color = '#2ca02c'  # Green
class1_color = '#d62728'  # Red
combined_color = '#1f77b4'  # Blue
test_color = '#ff7f0e'   # Orange

x_pos = np.arange(1, 13)  # Month positions
width = 0.35  # Width of each box pair

for idx, feature_base in enumerate(selected_features):
    ax = axes2[idx]
    
    # Get the column names for this feature
    column_list = complete_features[feature_base]
    
    # Prepare data for box plots
    train_data_by_month_class0 = []
    train_data_by_month_class1 = []
    train_data_by_month_combined = []
    test_data_by_month = []
    
    for month_idx, month in enumerate(months):
        col_name = column_list[month_idx]
        
        # Get data for this month
        train_vals_class0 = train_features_class0[col_name].dropna().values
        train_vals_class1 = train_features_class1[col_name].dropna().values
        train_vals_combined = train_features_combined[col_name].dropna().values
        test_vals = test_features[col_name].dropna().values
        
        train_data_by_month_class0.append(train_vals_class0)
        train_data_by_month_class1.append(train_vals_class1)
        train_data_by_month_combined.append(train_vals_combined)
        test_data_by_month.append(test_vals)
    
    # Create box plots for train data class 0 (shifted left)
    box_train0 = ax.boxplot(train_data_by_month_class0, 
                           positions=x_pos - width,
                           widths=width,
                           patch_artist=True,
                           boxprops=dict(facecolor=class0_color, alpha=0.7),
                           medianprops=dict(color='black'),
                           whiskerprops=dict(color='black'),
                           capprops=dict(color='black'),
                           flierprops=dict(marker='o', markerfacecolor=class0_color, 
                                           markersize=3, alpha=0.5))
    
    # Create box plots for train data class 1 (centered)
    box_train1 = ax.boxplot(train_data_by_month_class1,
                           positions=x_pos,
                           widths=width,
                           patch_artist=True,
                           boxprops=dict(facecolor=class1_color, alpha=0.7),
                           medianprops=dict(color='black'),
                           whiskerprops=dict(color='black'),
                           capprops=dict(color='black'),
                           flierprops=dict(marker='s', markerfacecolor=class1_color, 
                                           markersize=3, alpha=0.5))
    
    # Create box plots for train data combined (shifted right)
    box_train_combined = ax.boxplot(train_data_by_month_combined,
                           positions=x_pos + width,
                           widths=width,
                           patch_artist=True,
                           boxprops=dict(facecolor=combined_color, alpha=0.7),
                           medianprops=dict(color='black'),
                           whiskerprops=dict(color='black'),
                           capprops=dict(color='black'),
                           flierprops=dict(marker='^', markerfacecolor=combined_color, 
                                           markersize=3, alpha=0.5))
    
    # Create box plots for test data (further right)
    box_test = ax.boxplot(test_data_by_month,
                          positions=x_pos + 2*width,
                          widths=width,
                          patch_artist=True,
                          boxprops=dict(facecolor=test_color, alpha=0.7),
                          medianprops=dict(color='black'),
                          whiskerprops=dict(color='black'),
                          capprops=dict(color='black'),
                          flierprops=dict(marker='D', markerfacecolor=test_color, 
                                          markersize=3, alpha=0.5))
    
    # Customize subplot
    ax.set_title(f'{feature_base}', fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Month', fontsize=10)
    ax.set_ylabel('Value', fontsize=10)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(month_names, rotation=45, ha='right', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Add legend only to first subplot
    if idx == 0:
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor=class0_color, alpha=0.7, label='Train (class 0 - no aquaculture)'),
            Patch(facecolor=class1_color, alpha=0.7, label='Train (class 1 - aquaculture)'),
            Patch(facecolor=combined_color, alpha=0.7, label='Train (combined)'),
            Patch(facecolor=test_color, alpha=0.7, label='Test')
        ]
        ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
    
    # Progress indicator
    if (idx + 1) % 3 == 0 or (idx + 1) == len(selected_features):
        print(f"  Created box plot {idx + 1}/{len(selected_features)}")

# Hide unused subplots
for idx in range(len(selected_features), len(axes2)):
    axes2[idx].set_visible(False)

plt.suptitle('Monthly Feature Distributions: Train (Classes 0,1,Combined) vs Test\n(Box plots show distribution of values across months)',
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# Calculate overall statistics to quantify differences between train and test
# We'll compute average monthly values for each feature and compare

print("Computing summary statistics for feature comparison...")

# Storage for feature-level comparison
feature_comparison = []

for feature_base, stats_dict in monthly_stats.items():
    # Get annual average (mean across months) for train (using combined) and test
    train_means = np.array(stats_dict['train_combined']['mean'])
    test_means = np.array(stats_dict['test']['mean'])
    
    # Calculate mean across months (ignoring NaN)
    train_annual_mean = np.nanmean(train_means)
    test_annual_mean = np.nanmean(test_means)
    
    # Calculate standard deviation across months (temporal variability)
    train_temporal_std = np.nanstd(train_means, ddof=1)
    test_temporal_std = np.nanstd(test_means, ddof=1)
    
    # Calculate difference
    mean_diff = test_annual_mean - train_annual_mean
    if train_annual_mean != 0:
        mean_diff_pct = (mean_diff / abs(train_annual_mean)) * 100
    else:
        mean_diff_pct = np.nan
    
    if not (np.isnan(train_annual_mean) and np.isnan(test_annual_mean)):
        feature_comparison.append({
            'feature': feature_base,
            'train_annual_mean': train_annual_mean,
            'test_annual_mean': test_annual_mean,
            'mean_difference': mean_diff,
            'mean_difference_pct': mean_diff_pct,
            'train_temporal_std': train_temporal_std,
            'test_temporal_std': test_temporal_std
        })

# Convert to DataFrame
comparison_df = pd.DataFrame(feature_comparison)

print(f"Comparison of annual mean values for {len(comparison_df)} features:")
display(comparison_df.head(10))

# Summary statistics
print(f"Summary of differences:")
valid_diffs = comparison_df['mean_difference'].dropna()
if len(valid_diffs) > 0:
    print(f"  Mean absolute difference: {np.abs(valid_diffs).mean():.4f}")
    print(f"  Median absolute difference: {np.abs(valid_diffs).median():.4f}")
    print(f"  Max absolute difference: {np.abs(valid_diffs).max():.4f}")
    
    valid_pct = comparison_df['mean_difference_pct'].dropna()
    if len(valid_pct) > 0:
        print(f"  Mean absolute % difference: {np.abs(valid_pct).mean():.2f}%")
        print(f"  Median absolute % difference: {np.abs(valid_pct).median():.2f}%")
    
    # Count features with substantial differences
    large_diff = (np.abs(comparison_df['mean_difference_pct']) > 10).sum()
    print(f"  Features with >10% difference in annual mean: {large_diff} out of {len(comparison_df)}")
else:
    print("  No valid differences to report")

# Show top 5 features with largest percentage differences
if len(comparison_df) > 0 and 'mean_difference_pct' in comparison_df.columns:
    sorted_df = comparison_df.reindex(
        comparison_df['mean_difference_pct'].abs().sort_values(ascending=False).index
    )
    print(f"Top 5 features with largest percentage differences:")
    display(sorted_df[['feature', 'mean_difference_pct']].head())

---

## Summary

This analysis examined monthly patterns in features to compare training and test datasets, 
with training data analyzed separately by class (0=no aquaculture, 1=aquaculture) and combined.

### Key Findings

1. **Monthly Trends**: The error bar plots show how each feature's mean value changes across months 
   for training data (by class and combined) and test datasets, with error bars representing standard error of the mean.

2. **Distribution Patterns**: The box plots reveal the distribution of values for each feature 
   across months, allowing comparison of central tendency, spread, and potential outliers
   between train (by class and combined) and test data.

3. **Overall Differences**: The summary statistics quantify average differences between datasets
   in terms of annual mean values and temporal variability patterns.

### Observations

- Look for consistent patterns or divergences between train (by class) and test across months
- Identify features with significant seasonal variations
- Note any months where train and test distributions show substantial differences
- Consider whether observed differences might impact model generalization
- Compare the two classes (0 and 1) to see how they differ from each other and from the test data

### Next Steps

- Investigate features with large train-test differences further
- Consider whether seasonal adjustment or normalization might improve model performance
- Examine whether specific months or seasons drive the observed differences
- Use these insights to inform feature engineering or data preprocessing strategies
- Consider whether modeling approaches should account for class-specific seasonal patterns